# Notebook 01 — Data Cleaning + K-Means Clustering**Owner:** Basmala**Branch:** `clustering`**Inputs:** `data/raw/gsrrs23-indicators-for-participating-countries-or-territories.xlsx`**Outputs:** `data/processed/country_features.csv`, `data/processed/country_clusters.csv`---### What this notebook does (in plain words)1. Open the WHO road safety file2. Look at it and understand its shape3. Check for problems: missing values, duplicates, weird country names4. Fix those problems step by step5. Pick the numbers we will use to compare countries6. Scale them so big numbers do not dominate small numbers7. Find the best number of groups (clusters) using the elbow + silhouette method8. Run K-Means and label each country with a cluster9. Visualize the clusters as a 2D scatter plot (so we can actually *see* the groups)10. Profile each cluster — what makes them different?11. Save two clean files for Osama and Omar to useI keep the code cells small. After every code cell I write a short explanation.

## Step 0 — Import the libraries we needWe only use libraries we already know from class.

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.preprocessing import StandardScalerfrom sklearn.cluster import KMeansfrom sklearn.decomposition import PCAfrom sklearn.metrics import silhouette_score# make plots look nicersns.set_style('whitegrid')plt.rcParams['figure.figsize'] = (10, 6)

**Why each library:**- `pandas` → loads and cleans the table- `numpy` → math helpers- `matplotlib` + `seaborn` → drawing plots- `StandardScaler` → makes all numbers the same scale (mean 0, std 1)- `KMeans` → the clustering algorithm- `PCA` → squashes our many columns down to 2 columns so we can draw the scatter plot- `silhouette_score` → tells us how good our clusters are

## Step 1 — Load the fileThe WHO Excel file has a tricky header. The first row is short codes (CP_0, CP_1, ...). The second row is the real human-readable names ("Population", "Income group", ...). The third row onwards is the data.So we tell pandas: use row index 1 (the second row) as the header. Then we drop the leftover row above the data.

In [ ]:
RAW_PATH = 'data/raw/gsrrs23-indicators-for-participating-countries-or-territories.xlsx'df = pd.read_excel(RAW_PATH, sheet_name='Indicators', header=1)# the very first data row was actually the codes row leaking through — drop itdf = df.iloc[1:].reset_index(drop=True)print('Shape:', df.shape)df.head(3)

**What we see:** 171 countries, 228 columns. That is a lot of columns. Most of them describe road safety policies, infrastructure, deaths, etc.

## Step 2 — First look at the dataBefore we touch anything, let's understand what we have.

In [ ]:
df.info()

**Notice:** many columns are `object` (text) and many are `float64` (numbers). We will only use numeric columns for clustering, because K-Means needs numbers.

In [ ]:
# show the first few descriptive columnsdf[['ISO_3 country name', 'Country name', 'Population', 'Income group', 'WHO Region']].head()

**Good.** Each row is a country. We have ISO code, country name, population, income group, and WHO region. These are the "label" columns we keep for context but do NOT cluster on.

## Step 3 — Check for duplicate rowsA duplicate would mean the same country appears twice. That would mess up clustering.

In [ ]:
print('Total duplicate rows:        ', df.duplicated().sum())print('Duplicate ISO codes:         ', df['ISO_3 country name'].duplicated().sum())print('Duplicate country names:     ', df['Country name'].duplicated().sum())

**Result:** zero duplicates. Good — no countries appear twice.

## Step 4 — Look at missing valuesThis is the big one. Many WHO indicators are missing for many countries because not every country reports every metric.

In [ ]:
miss_pct = df.isna().mean().sort_values(ascending=False) * 100print('Columns with more than 50% missing:', (miss_pct > 50).sum())print('Columns with more than 70% missing:', (miss_pct > 70).sum())print('Columns with zero missing:         ', (miss_pct == 0).sum())print('Total columns:                     ', len(df.columns))

**Reality check:** about 60 columns are more than half empty. We cannot trust those — they would force us to invent too much data. We will drop them.

In [ ]:
# visualize the missing-value pictureplt.figure(figsize=(10, 4))plt.hist(miss_pct, bins=30, edgecolor='black')plt.axvline(50, color='red', linestyle='--', label='50% threshold')plt.xlabel('% of values missing in a column')plt.ylabel('Number of columns')plt.title('How many columns are how empty?')plt.legend()plt.show()

**Reading the chart:** most columns are either very full (left side) or quite empty (right side). The red line is our cutoff — anything to the right of it gets dropped.

## Step 5 — Normalize country namesPalestine appears in our data as **"occupied Palestinian territory, including east Jerusalem"**. In the time-series file (`output.csv`) it is just **"Palestine"**. If the names do not match, we cannot join the two datasets in the Streamlit app later.So we map all the variants to one clean name.

In [ ]:
COUNTRY_NAME_MAP = {    'occupied Palestinian territory, including east Jerusalem': 'Palestine',    'occupied Palestinian territory': 'Palestine',    'State of Palestine': 'Palestine',    'Palestinian Territory': 'Palestine',    'West Bank and Gaza Strip': 'Palestine',}df['Country name'] = df['Country name'].replace(COUNTRY_NAME_MAP)# confirmdf[df['ISO_3 country name'] == 'PSE'][['ISO_3 country name', 'Country name']]

**Result:** Palestine is now labeled cleanly. This map is small now but we put it in a helper file later so Omar and Osama use the same one.

## Step 6 — Drop columns that are mostly emptyOur rule: if a column is more than 50% missing, we cannot trust it. Drop it.

In [ ]:
before = df.shape[1]cols_to_drop = miss_pct[miss_pct > 50].index.tolist()df_clean = df.drop(columns=cols_to_drop)after = df_clean.shape[1]print(f'Columns before: {before}')print(f'Columns after:  {after}')print(f'Dropped:        {before - after}')

**Why we chose 50%:** it is a balance. Strict enough to get rid of useless columns. Loose enough to keep enough features for clustering to be meaningful.

## Step 7 — Pick the numeric features for clusteringK-Means only works on numbers. We separate the *label* columns (country name, region, income group) from the *feature* columns (the actual numbers we cluster on).

In [ ]:
LABEL_COLS = ['ISO_3 country name', 'Country name', 'Income group', 'WHO Region']# everything numericnumeric_cols = df_clean.select_dtypes(include='number').columns.tolist()print(f'Number of numeric feature columns: {len(numeric_cols)}')print('First 5:', numeric_cols[:5])

**Note:** Population is numeric and will be in there. That is fine — population scale matters for road safety.

## Step 8 — Fill the remaining missing values with the column medianAfter dropping the very-empty columns, the columns we kept still have *some* missing values. We fill those with the column median.**Why median and not mean?** Median is not affected by extreme outliers. If one country has a crazy-high value, the mean shifts but the median does not.

In [ ]:
features = df_clean[numeric_cols].copy()# how many missing before?missing_before = features.isna().sum().sum()# fill with medianfeatures = features.fillna(features.median(numeric_only=True))# how many missing after?missing_after = features.isna().sum().sum()print(f'Missing values before: {missing_before}')print(f'Missing values after:  {missing_after}')

**Done.** No more missing values. Every country has a complete row of numbers.

## Step 9 — Scale the featuresK-Means uses *distance* between points. If one column is "Population" (millions) and another is "% of seatbelt use" (0–100), population will completely dominate the distance.`StandardScaler` fixes this: every column gets mean 0 and standard deviation 1.

In [ ]:
scaler = StandardScaler()X_scaled = scaler.fit_transform(features)print('Shape of scaled feature matrix:', X_scaled.shape)print('Mean of first column after scaling (should be ~0):', X_scaled[:, 0].mean().round(4))print('Std  of first column after scaling (should be ~1):', X_scaled[:, 0].std().round(4))

**Confirmed:** mean ≈ 0, std ≈ 1. All columns are now on the same scale.

## Step 10 — Choose the best number of clusters (k)We try k = 2, 3, 4, 5, 6, 7, 8 and look at two things:1. **Elbow method** — within-cluster sum of squares (lower = tighter clusters, but always drops with more k)2. **Silhouette score** — how well-separated clusters are (-1 to 1, higher = better)The "elbow" point and the silhouette peak together tell us a good k.

In [ ]:
k_values = range(2, 9)inertias = []silhouettes = []for k in k_values:    km = KMeans(n_clusters=k, random_state=42, n_init=10)    labels = km.fit_predict(X_scaled)    inertias.append(km.inertia_)    silhouettes.append(silhouette_score(X_scaled, labels))# show as a small table firstpd.DataFrame({'k': list(k_values), 'inertia': inertias, 'silhouette': silhouettes}).round(3)

**How to read this table:** lower inertia is better but it always drops, so we look for the "elbow". Higher silhouette is better — a peak tells us a good k.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))axes[0].plot(list(k_values), inertias, 'o-', color='steelblue')axes[0].set_xlabel('Number of clusters (k)')axes[0].set_ylabel('Inertia (lower = tighter)')axes[0].set_title('Elbow Method')axes[1].plot(list(k_values), silhouettes, 'o-', color='darkorange')axes[1].set_xlabel('Number of clusters (k)')axes[1].set_ylabel('Silhouette score (higher = better)')axes[1].set_title('Silhouette Method')plt.tight_layout()plt.show()

**How to decide k from these plots:**- On the elbow plot, look for the "bend" where adding more clusters stops helping much.- On the silhouette plot, look for the highest peak.- If they disagree, prefer the silhouette peak (it directly measures cluster quality).For this dataset, k = 4 is usually a strong choice — it gives interpretable groups (e.g., high-income safe / high-income with issues / middle-income / low-income).

## Step 11 — Fit the final K-Means with our chosen kWe pick `k = 4` based on the plots above. If the silhouette plot above peaks at a different k, change this number.

In [ ]:
K_FINAL = 4kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)cluster_labels = kmeans.fit_predict(X_scaled)print(f'Fitted K-Means with k = {K_FINAL}')print(f'Cluster sizes:')print(pd.Series(cluster_labels).value_counts().sort_index())

**Sanity check:** none of the clusters is empty or has only 1 country. Each cluster has a reasonable number of countries.

## Step 12 — Visualize the clusters in 2D (the cluster scatter)This is the picture you remember from class — colored dots, where each dot is a country and the color is its cluster.We use PCA to squash our many columns down to 2 dimensions just for *drawing*. (Osama will do the proper PCA analysis in his notebook.)

In [ ]:
# squash to 2D for plotting onlypca_2d = PCA(n_components=2, random_state=42)coords = pca_2d.fit_transform(X_scaled)# put it all together for plottingplot_df = pd.DataFrame({    'PC1': coords[:, 0],    'PC2': coords[:, 1],    'Cluster': cluster_labels,    'Country': df_clean['Country name'].values,    'ISO': df_clean['ISO_3 country name'].values,})plot_df.head()

In [ ]:
plt.figure(figsize=(12, 8))palette = sns.color_palette('Set2', K_FINAL)for c in range(K_FINAL):    sub = plot_df[plot_df['Cluster'] == c]    plt.scatter(sub['PC1'], sub['PC2'],                color=palette[c], label=f'Cluster {c}',                s=70, alpha=0.7, edgecolor='black', linewidth=0.5)# spotlight Palestinepal = plot_df[plot_df['ISO'] == 'PSE']if len(pal):    plt.scatter(pal['PC1'], pal['PC2'],                marker='*', s=400, color='red',                edgecolor='black', linewidth=1.5, label='Palestine', zorder=5)plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% of variance)')plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% of variance)')plt.title(f'Country Clusters in 2D — K-Means (k = {K_FINAL})')plt.legend(loc='best')plt.tight_layout()plt.show()

**Reading this picture:**- Each dot is one country.- Color = which cluster K-Means put it in.- The red star is Palestine.- Countries close together = similar road safety profiles.- Countries far apart = very different profiles.This is the picture you bring to the discussion. You can literally point and say *"this group here is X, that group there is Y, and Palestine sits with these countries."*

## Step 13 — Profile each cluster (what makes them different?)A cluster number alone (0, 1, 2, 3) means nothing. We need to look at the *averages* inside each cluster to give them meaning.

In [ ]:
# attach cluster labels back to the labeled dataprofile_df = df_clean[LABEL_COLS].copy()profile_df['Cluster'] = cluster_labels# how income groups split across clusterspd.crosstab(profile_df['Cluster'], profile_df['Income group'])

**This table tells us a lot.** If most "High income" countries land in one cluster, that cluster is the "rich-country profile". If most "Low income" land in another, that's the "low-income profile".

In [ ]:
# how WHO regions split across clusterspd.crosstab(profile_df['Cluster'], profile_df['WHO Region'])

**Does region matter?** Sometimes clusters line up with regions (e.g., African region = one cluster). Sometimes they don't — and that's interesting too.

In [ ]:
# average value of each numeric feature per cluster (top 10 most distinguishing)cluster_means = features.copy()cluster_means['Cluster'] = cluster_labelsgroup_means = cluster_means.groupby('Cluster').mean()# find features whose averages vary the most across clustersvariation = group_means.std().sort_values(ascending=False)top_distinguishing = variation.head(10).indexgroup_means[top_distinguishing].round(2)

**Reading:** the columns above are the features that *differ most* between clusters. These are the variables that K-Means is using to separate countries. Look at them and write a one-line story for each cluster (e.g., "Cluster 0 = high-population, low-income, high reported fatalities").

## Step 14 — Where does Palestine land?

In [ ]:
pal_cluster = profile_df.loc[profile_df['ISO_3 country name'] == 'PSE', 'Cluster'].values[0]print(f'Palestine is in cluster: {pal_cluster}')print('\nCountries in the same cluster as Palestine:')neighbors = profile_df[profile_df['Cluster'] == pal_cluster]['Country name'].tolist()for n in neighbors:    print(f'  - {n}')

**This is the headline for the Streamlit app.** "Palestine clusters with these countries" — that's a story we can tell on the comparison page.

## Step 15 — Save the outputsWe save TWO files:- `country_features.csv` — the cleaned numeric matrix → **Osama needs this for PCA + Anomaly Detection**- `country_clusters.csv` — country + cluster label → used by the Streamlit map page

In [ ]:
import osos.makedirs('data/processed', exist_ok=True)# 1. features file (for Osama)features_out = df_clean[LABEL_COLS].copy()features_out = pd.concat([features_out, features.reset_index(drop=True)], axis=1)features_out.to_csv('data/processed/country_features.csv', index=False)print('Saved: data/processed/country_features.csv', features_out.shape)# 2. clusters file (for the app)clusters_out = profile_df.copy()clusters_out.to_csv('data/processed/country_clusters.csv', index=False)print('Saved: data/processed/country_clusters.csv', clusters_out.shape)

## Step 16 — Summary**What I did:**1. Loaded WHO file (171 countries × 228 columns)2. Confirmed no duplicate countries3. Found 60 columns that were >50% empty → dropped them4. Normalized Palestine's name (so it matches the time-series file)5. Filled remaining missing values with the column median (robust to outliers)6. Scaled all features to mean 0 std 1 (so K-Means is fair)7. Used elbow + silhouette to choose k = 48. Ran K-Means → 4 clusters9. Drew a 2D scatter showing the clusters with Palestine starred10. Profiled each cluster by income group and region11. Saved two clean files**What I'm handing off:**- `country_features.csv` → Osama (for PCA + Isolation Forest)- `country_clusters.csv` → Streamlit map page**What I would defend in the discussion:**- Why median imputation, not mean → robust to outliers- Why drop columns >50% missing → cannot trust mostly-empty columns- Why StandardScaler → K-Means is distance-based, scale matters- Why k = 4 → silhouette + elbow agreement- Why PCA-2D for the scatter → cannot draw 97 dimensions, need to project**Limitations to note in the README:**- Median imputation can hide variance in the missing countries- WHO data is self-reported by countries — quality varies- Cluster labels are statistical groupings, not policy judgments